# Sales Data Engineering Pipeline (Bronze -> Silver -> Gold)

## Overview & Architecture
This Databricks notebook implements an end-to-end PySpark ETL pipeline following a Test-Driven Development (TDD) and Medallion Architecture approach:
1. Transformation & Cleaning Functions: Modular PySpark functions handling data quality anomalies (noise characters in customer names, #ERROR! phone numbers, non-breaking spaces in product names, multi-line addresses, date parsing, and deduplication).
2. Unit Tests (unittest): Automated test suite validating edge cases before pipeline execution.
3. Bronze Layer (Raw Tables): Ingests Products_De22po.csv, Customer_xTMA1_.xlsx, and Orders_J9zDfj.json into Delta tables (raw_products, raw_customers, raw_orders).
4. Silver Layer (Enriched Tables): Builds enriched_customers, enriched_products, and the master enriched_orders table with Profit rounded to 2 decimal places, customer details (Customer Name, Country), and product hierarchy (Product Category, Product Sub Category).
5. **Gold Layer (Aggregate Table & SQL Reporting): Creates agg_profit_summary and queries profit aggregates via %sql.

### Step 1: Configuration & Imports

In [ ]:
import re
import unittest
import pandas as pd
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType
from pyspark.sql.window import Window

# Widgets for configurable input paths in Databricks
dbutils.widgets.text("products_path", "/FileStore/tables/Products.csv", "Products CSV Path")
dbutils.widgets.text("customers_path", "/FileStore/tables/Customer.xlsx", "Customers Excel Path")
dbutils.widgets.text("orders_path", "/FileStore/tables/Orders.json", "Orders JSON Path")

PRODUCTS_PATH = dbutils.widgets.get("products_path")
CUSTOMERS_PATH = dbutils.widgets.get("customers_path")
ORDERS_PATH = dbutils.widgets.get("orders_path")

### Step 2: Modular PySpark Cleaning, Enrichment & Aggregation Functions

In [ ]:
def sanitize_column_names(df: DataFrame) -> DataFrame:
    """Sanitizes column names to be Delta Lake compatible."""
    for col_name in df.columns:
        clean_col = re.sub(r"[ ,;{}()\n\t=-]+", "_", col_name).strip("_")
        df = df.withColumnRenamed(col_name, clean_col)
    return df

def clean_products(df: DataFrame) -> DataFrame:
    """
    Cleans raw product data:
    - Trims whitespace and replaces non-breaking spaces (\u00A0).
    - Casts Price per product to Double.
    - Deduplicates by Product ID to prevent fan-out joins across multiple states.
    """
    window_spec = Window.partitionBy("Product ID").orderBy(F.col("Price per product").desc())
    return (
        df.withColumn("Product ID", F.upper(F.trim(F.col("Product ID"))))
          .withColumn("Category", F.trim(F.col("Category")))
          .withColumn("Sub-Category", F.trim(F.col("Sub-Category")))
          .withColumn(
              "Product Name",
              F.trim(F.regexp_replace(F.col("Product Name"), r"\u00A0", " "))
          )
          .withColumn("Price per product", F.col("Price per product").cast(DoubleType()))
          .filter(F.col("Product ID").isNotNull())
          .withColumn("rn", F.row_number().over(window_spec))
          .filter(F.col("rn") == 1)
          .drop("rn", "State")
    )

def clean_customers(df: DataFrame) -> DataFrame:
    """
    Cleans raw customer data:
    - Strips digits/special symbols from Customer Name (e.g., 'Gary567 Hansen', 'Pete@#$ Takahito').
    - Replaces null or blank names with 'Unknown Customer'.
    - Nullifies '#ERROR!' or invalid short phone numbers.
    - Flattens multi-line addresses and deduplicates by Customer ID.
    """
    cleaned_name = F.regexp_replace(F.col("Customer Name"), r"[^a-zA-Z\s\-\']", "")
    cleaned_name = F.trim(F.regexp_replace(cleaned_name, r"\s+", " "))

    return (
        df.withColumn("Customer ID", F.upper(F.trim(F.col("Customer ID"))))
          .withColumn(
              "Customer Name",
              F.when(F.col("Customer Name").isNull() | (cleaned_name == ""), F.lit("Unknown Customer"))
               .otherwise(cleaned_name)
          )
          .withColumn("email", F.lower(F.trim(F.col("email"))))
          .withColumn(
              "phone",
              F.when((F.col("phone") == "#ERROR!") | (F.length(F.trim(F.col("phone"))) < 7), F.lit(None))
               .otherwise(F.trim(F.col("phone")))
          )
          .withColumn("address", F.trim(F.regexp_replace(F.col("address"), r"[\r\n]+", ", ")))
          .withColumn("Segment", F.trim(F.col("Segment")))
          .withColumn("Country", F.trim(F.col("Country")))
          .withColumn("City", F.trim(F.col("City")))
          .withColumn("State", F.trim(F.col("State")))
          .withColumn("Region", F.trim(F.col("Region")))
          .filter(F.col("Customer ID").isNotNull())
          .dropDuplicates(["Customer ID"])
    )

def clean_orders(df: DataFrame) -> DataFrame:
    """
    Cleans raw order data:
    - Parses 'd/M/yyyy' date strings into DateType and extracts Order Year.
    - Normalizes Customer ID and Product ID casing for reliable joins.
    - Casts numeric fields and rounds Profit to 2 decimal places.
    """
    return (
        df.withColumn("Order ID", F.trim(F.col("Order ID")))
          .withColumn("Customer ID", F.upper(F.trim(F.col("Customer ID"))))
          .withColumn("Product ID", F.upper(F.trim(F.col("Product ID"))))
          .withColumn("Order Date", F.to_date(F.col("Order Date"), "d/M/yyyy"))
          .withColumn("Ship Date", F.to_date(F.col("Ship Date"), "d/M/yyyy"))
          .withColumn("Order Year", F.year(F.col("Order Date")))
          .withColumn("Quantity", F.col("Quantity").cast(IntegerType()))
          .withColumn("Price", F.col("Price").cast(DoubleType()))
          .withColumn("Discount", F.col("Discount").cast(DoubleType()))
          .withColumn("Profit", F.round(F.col("Profit").cast(DoubleType()), 2))
          .filter(F.col("Order ID").isNotNull() & F.col("Order Date").isNotNull())
          .dropDuplicates(["Row ID"])
    )

def build_enriched_orders(
    orders_df: DataFrame, customers_df: DataFrame, products_df: DataFrame
) -> DataFrame:
    """
    Creates the master enriched orders table using broadcast joins.
    Includes Profit rounded to 2 decimals, Customer Name & Country, and Product Category & Sub Category.
    """
    cust_dim = customers_df.select(
        F.col("Customer ID"),
        F.col("Customer Name"),
        F.col("Country"),
        F.col("Segment"),
        F.col("Region")
    )

    prod_dim = products_df.select(
        F.col("Product ID"),
        F.col("Category").alias("Product Category"),
        F.col("Sub-Category").alias("Product Sub Category"),
        F.col("Product Name")
    )

    return (
        orders_df.join(F.broadcast(cust_dim), on="Customer ID", how="left")
                 .join(F.broadcast(prod_dim), on="Product ID", how="left")
                 .withColumn("Customer Name", F.coalesce(F.col("Customer Name"), F.lit("Unknown Customer")))
                 .withColumn("Country", F.coalesce(F.col("Country"), F.lit("Unknown")))
                 .withColumn("Product Category", F.coalesce(F.col("Product Category"), F.lit("Uncategorized")))
                 .withColumn("Product Sub Category", F.coalesce(F.col("Product Sub Category"), F.lit("Uncategorized")))
                 .withColumn("Profit", F.round(F.col("Profit"), 2))
    )

def build_profit_aggregate_table(enriched_orders_df: DataFrame) -> DataFrame:
    """Aggregates profit by Year, Product Category, Product Sub Category, and Customer."""
    return (
        enriched_orders_df.groupBy(
            F.col("Order Year").alias("Year"),
            F.col("Product Category"),
            F.col("Product Sub Category"),
            F.col("Customer ID"),
            F.col("Customer Name")
        )
        .agg(F.round(F.sum("Profit"), 2).alias("Total_Profit"))
    )

### Step 3: Unit Tests (Test-Driven Development)

In [ ]:
class TestSalesPipeline(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.spark = spark

    def test_clean_customers_handles_noise_and_nulls(self):
        raw_data = [
            ("gh-14410", "Gary567 Hansen", "AUSTIN@GMAIL.COM", "123-456-7890", "123 St\nCity, IL", "Home Office", "United States", "Chicago", "Illinois", "60653", "Central"),
            ("PT-19090", "Pete@#$ Takahito", "pete@gmail.com", "#ERROR!", "456 Ave", "Consumer", "United States", "San Antonio", "Texas", "78207", "Central"),
            ("EM-13810", None, "tracy@gmail.com", "-6181", "789 Rd", "Corporate", "United States", "New York City", "New York", "10009", "East")
        ]
        cols = ["Customer ID", "Customer Name", "email", "phone", "address", "Segment", "Country", "City", "State", "Postal Code", "Region"]
        df = self.spark.createDataFrame(raw_data, cols)

        result_map = {row["Customer ID"]: row for row in clean_customers(df).collect()}

        self.assertEqual(result_map["GH-14410"]["Customer Name"], "Gary Hansen")
        self.assertEqual(result_map["GH-14410"]["email"], "austin@gmail.com")
        self.assertEqual(result_map["GH-14410"]["address"], "123 St, City, IL")
        self.assertEqual(result_map["PT-19090"]["Customer Name"], "Pete Takahito")
        self.assertIsNone(result_map["PT-19090"]["phone"])
        self.assertEqual(result_map["EM-13810"]["Customer Name"], "Unknown Customer")
        self.assertIsNone(result_map["EM-13810"]["phone"])

    def test_clean_products_deduplicates_and_strips_nbsp(self):
        raw_data = [
            ("TEC-AC-10004659", "Technology", "Accessories", "Imation\u00A0Secure+ Drive", "Oklahoma", 72.99),
            ("TEC-AC-10004659", "Technology", "Accessories", "Imation Secure+ Drive", "Texas", 72.99)
        ]
        cols = ["Product ID", "Category", "Sub-Category", "Product Name", "State", "Price per product"]
        df = self.spark.createDataFrame(raw_data, cols)

        result = clean_products(df).collect()
        self.assertEqual(len(result), 1)
        self.assertEqual(result[0]["Product Name"], "Imation Secure+ Drive")

    def test_enrichment_and_aggregation(self):
        orders_df = clean_orders(self.spark.createDataFrame([
            (1, "CA-2016-122581", "21/8/2016", "25/8/2016", "Standard Class", "jk-15370", "FUR-CH-10002961", 7, 573.17, 0.3, 63.694),
            (2, "CA-2016-999999", "22/8/2016", "26/8/2016", "Standard Class", "MISSING-ID", "MISSING-PROD", 1, 10.0, 0.0, 2.556)
        ], ["Row ID", "Order ID", "Order Date", "Ship Date", "Ship Mode", "Customer ID", "Product ID", "Quantity", "Price", "Discount", "Profit"]))

        cust_df = clean_customers(self.spark.createDataFrame([
            ("JK-15370", "Jay Kimmel", "a@b.com", "1234567890", "Addr", "Consumer", "United States", "NYC", "NY", "10035", "East")
        ], ["Customer ID", "Customer Name", "email", "phone", "address", "Segment", "Country", "City", "State", "Postal Code", "Region"]))

        prod_df = clean_products(self.spark.createDataFrame([
            ("FUR-CH-10002961", "Furniture", "Chairs", "Leather Task Chair", "NY", 81.882)
        ], ["Product ID", "Category", "Sub-Category", "Product Name", "State", "Price per product"]))

        enriched = build_enriched_orders(orders_df, cust_df, prod_df)
        rows = {r["Row ID"]: r for r in enriched.collect()}

        self.assertEqual(rows[1]["Customer Name"], "Jay Kimmel")
        self.assertEqual(rows[1]["Product Category"], "Furniture")
        self.assertEqual(rows[1]["Profit"], 63.69)

        self.assertEqual(rows[2]["Customer Name"], "Unknown Customer")
        self.assertEqual(rows[2]["Product Category"], "Uncategorized")
        self.assertEqual(rows[2]["Profit"], 2.56)

        agg_df = build_profit_aggregate_table(enriched)
        self.assertEqual(agg_df.count(), 2)

suite = unittest.TestLoader().loadTestsFromTestCase(TestSalesPipeline)
runner = unittest.TextTestRunner(verbosity=2)
test_result = runner.run(suite)
assert test_result.wasSuccessful(), "Unit tests failed! Halting pipeline execution."

### Step 4: Bronze Layer — Ingest Raw Source Datasets into Delta Tables

In [ ]:
try:
    # 1. Raw Products (CSV)
    raw_products_df = (
        spark.read.format("csv")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(PRODUCTS_PATH)
    )

    # 2. Raw Customers (Excel)
    local_excel_path = CUSTOMERS_PATH.replace("/FileStore", "/dbfs/FileStore") if CUSTOMERS_PATH.startswith("/FileStore") else CUSTOMERS_PATH
    pdf_customers = pd.read_excel(local_excel_path, dtype=str)
    raw_customers_df = spark.createDataFrame(pdf_customers)

    # 3. Raw Orders (JSON)
    raw_orders_df = (
        spark.read.format("json")
        .option("multiLine", "true")
        .load(ORDERS_PATH)
    )

    # Save Bronze Delta Tables
    sanitize_column_names(raw_products_df).write.format("delta").mode("overwrite").saveAsTable("raw_products")
    sanitize_column_names(raw_customers_df).write.format("delta").mode("overwrite").saveAsTable("raw_customers")
    sanitize_column_names(raw_orders_df).write.format("delta").mode("overwrite").saveAsTable("raw_orders")
    print("Bronze tables successfully created: raw_products, raw_customers, raw_orders")

except Exception as e:
    raise RuntimeError(f"Error during Bronze table ingestion: {str(e)}")

### Step 5: Silver & Gold Layers — Enriched Tables & Profit Aggregations

In [ ]:
try:
    # Clean source DataFrames
    enriched_products_df = clean_products(raw_products_df)
    enriched_customers_df = clean_customers(raw_customers_df)
    cleaned_orders_df = clean_orders(raw_orders_df)

    # Persist Enriched Dimension Tables
    sanitize_column_names(enriched_customers_df).write.format("delta").mode("overwrite").saveAsTable("enriched_customers")
    sanitize_column_names(enriched_products_df).write.format("delta").mode("overwrite").saveAsTable("enriched_products")

    # Persist Enriched Orders Master Table
    enriched_orders_df = build_enriched_orders(
        cleaned_orders_df, enriched_customers_df, enriched_products_df
    )
    sanitize_column_names(enriched_orders_df).write.format("delta").mode("overwrite").saveAsTable("enriched_orders")

    # Persist Gold Aggregate Table
    agg_profit_df = build_profit_aggregate_table(enriched_orders_df)
    sanitize_column_names(agg_profit_df).write.format("delta").mode("overwrite").saveAsTable("agg_profit_summary")
    print("Silver and Gold tables successfully created: enriched_customers, enriched_products, enriched_orders, agg_profit_summary")

except Exception as e:
    raise RuntimeError(f"Error during Silver/Gold transformation: {str(e)}")

### Step 6: SQL Queries for Profit Aggregations

In [ ]:
%sql
-- 1. Profit by Year
SELECT 
    Year,
    ROUND(SUM(Total_Profit), 2) AS Profit
FROM agg_profit_summary
GROUP BY Year
ORDER BY Year;

In [ ]:
%sql
-- 2. Profit by Year + Product Category
SELECT 
    Year,
    Product_Category,
    ROUND(SUM(Total_Profit), 2) AS Profit
FROM agg_profit_summary
GROUP BY Year, Product_Category
ORDER BY Year, Product_Category;

In [ ]:
%sql
-- 3. Profit by Customer
SELECT 
    Customer_ID,
    Customer_Name,
    ROUND(SUM(Total_Profit), 2) AS Profit
FROM agg_profit_summary
GROUP BY Customer_ID, Customer_Name
ORDER BY Profit DESC;

In [ ]:
%sql
-- 4. Profit by Customer + Year
SELECT 
    Customer_ID,
    Customer_Name,
    Year,
    ROUND(SUM(Total_Profit), 2) AS Profit
FROM agg_profit_summary
GROUP BY Customer_ID, Customer_Name, Year
ORDER BY Customer_Name, Year;